# 6c · Single-call end-to-end reified extraction  (BASELINE)

The **end-to-end baseline** for the staged event-centric pipeline (nb2→nb4→nb6b). Instead of extracting
entities, typing/deduping them, and *then* extracting n-ary nodes + relations over that frozen entity
set, this notebook does it all in **ONE Gemini call from the raw TEXT**: entities (with open free-form
types), n-ary nodes (Event / Observation / Claim), and open `relations`.

It exists to answer *"is the staged decomposition actually better than doing everything in one call?"* —
run it alongside the staged path and compare (context completeness, fragmentation, predicate/type
economy, QA faithfulness, entity precision/recall, truncation rate).

Output is the **same shape** as nb6b's `nary_graph.json`, written to `results/<id>/nary_graph_e2e.json`,
so it flows through the SAME downstream:
- **nb7b** canonicalizes its relations — add `"nary_graph_e2e.json"` to nb7b's input glob/name.
- **nb8** renders it — add `"nary_graph_e2e.json"` to `REL_FILES`.

(Entity *types* here are open too; canonicalizing them is the nb3/nb4-style step — the same gap the
staged n-ary path has. This notebook is the extraction baseline; canonicalization is shared downstream.)

In [19]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs (results/ now has sub-folders like
# two_staged/1_entities/); guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [ ]:
# --- Config -------------------------------------------------------------------------------
from pathlib import Path

# ---- WHICH MODEL'S RESULTS ---------------------------------------------------------------
# Everything is read from and written to results/<RESULTS_MODEL>/. Choose any subfolder that
# exists there: "gemini-2.5-flash", "qwen3-14b", "qwen3-32b", "llama33-70b", "qwen3-32b-think" ... (the names are
# the BACKENDS keys in notebooks/llm_backend.py).
RESULTS_MODEL = "qwen3-32b-think"

# Which model runs THIS notebook's LLM calls. Usually the same one; set it to
# "gemini-2.5-flash" to process another model's outputs with Gemini (e.g. as an eval judge).
# Anything else needs a vLLM server + tunnel — see cluster/README.md.
BACKEND = RESULTS_MODEL
# ------------------------------------------------------------------------------------------

GEMINI_MODEL = "gemini-2.5-flash"          # overwritten by activate() in the next cell
RESULTS_DIR  = ROOT / "results" / RESULTS_MODEL
DATASET_DIR  = ROOT / "dataset/articles"   # reads <id>.txt (raw text — NO pre-extracted entities)

ARTICLE_ID   = "art1"                      # single article used by the demo cell below
SELECTED_ARTICLES = ["art1"]   # batch set
# SELECTED_ARTICLES = ["art1", "art2", "art4", "art5"]   # batch set

OUTPUT_NAME  = "e2e/nary_graph_dedup.json"       # written to <RESULTS_DIR>/<id>/  (distinct from the staged nary_graph.json)

# Articles are extracted independently — nothing is carried from one to the next — so the batch cell
# can run several at once. WORKERS is how many ARTICLES are in flight (each costs 2 LLM calls,
# extract + review), i.e. the concurrency the API sees. 1 = the old sequential loop; use it when
# debugging. Gemini paid tier: 4-8 is comfortable; free tier (10 RPM): 1-2. vLLM: 4-8.
WORKERS = 4

# Articles this run processes: those with a raw <id>.txt.
_available = sorted(p.stem for p in DATASET_DIR.glob("*.txt"))
if SELECTED_ARTICLES is None:
    ARTICLE_IDS = _available
else:
    missing = [a for a in SELECTED_ARTICLES if a not in _available]
    assert not missing, f"no article text for: {missing}"
    ARTICLE_IDS = list(SELECTED_ARTICLES)

print(f"{len(_available)} article texts available; this run will process {ARTICLE_IDS}")

18 article texts available; this run will process ['art1', 'art2', 'art4', 'art5']


In [21]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai json-repair openai

import os, json, time, re
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown, display
try:
    from json_repair import repair_json          # salvages malformed LLM JSON (unquoted numbers/dates, etc.)
except Exception:
    repair_json = None

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Paste your key in GEMINI_API_KEY above, or set it in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)


last_latency_s = None
LAST_USAGE = None          # token usage of the most recent call: {prompt_tokens, output_tokens, thoughts_tokens, total_tokens}

MAX_OUTPUT_TOKENS = None   # None -> no cap (model max). This call is BIG (entities+nodes+relations at once).
THINKING_BUDGET   = -1      # -1 dynamic (model decides) | 0 thinking OFF (whole budget -> JSON) | N cap thinking tokens

def _loads_lenient(text, resp=None):
    t = (text or "").strip()
    if t.startswith("```"):
        t = re.sub(r"^```[a-zA-Z]*\n?", "", t); t = re.sub(r"\n?```$", "", t).strip()
    # 1) strict, then 2) trailing-comma fix
    for cand in (t, re.sub(r",(\s*[}\]])", r"\1", t)):
        try:
            return json.loads(cand)
        except json.JSONDecodeError:
            pass
    # 3) json_repair fallback — fixes unquoted numbers/dates, missing values, bad escapes, etc.
    if repair_json is not None:
        try:
            fixed = repair_json(t)
            obj = json.loads(fixed)
            print(f"    [json_repair] salvaged a malformed response ({len(t)} chars)")
            return obj
        except Exception:
            pass
    # 4) give up — report the finish_reason, the exact offending spot, and dump the raw text
    try:
        json.loads(t)
    except json.JSONDecodeError as e:
        pos = getattr(e, "pos", 0)
        ctx = t[max(0, pos - 160): pos + 80]
        fr = None
        try: fr = resp.candidates[0].finish_reason
        except Exception: pass
        dbg = None
        try:
            dbg = RESULTS_DIR / "_debug_last_bad_response.json"; dbg.write_text(t)
        except Exception:
            pass
        truncated = str(fr).endswith("MAX_TOKENS")
        raise ValueError(
            f"Gemini JSON unparseable (finish_reason={fr}; {len(t)} chars; error at char {pos}). "
            + ("Looks TRUNCATED (MAX_TOKENS) — reduce input or split the call. "
               if truncated else
               "NOT truncated (finished normally) — a malformed token mid-JSON (often an unquoted number/date). ")
            + (f"raw response saved to {dbg}. " if dbg else "")
            + f"json error: {e}\n...context around the error (>>> marks the spot):\n"
            + f"{ctx[:160]!r} >>> {ctx[160:]!r}") from None

def _record_usage(resp, seconds=None):
    """Pull token counts off the response's usage_metadata into LAST_USAGE (None-safe). thoughts_tokens
    is the reasoning ('thinking') portion — nonzero once THINKING_BUDGET is -1/positive. The shared
    helper also keeps a per-thread copy (the global belongs to whichever call finished LAST, which in
    a parallel batch is not ours) and appends to the ledger behind nb_parallel.usage_report()."""
    global LAST_USAGE
    LAST_USAGE = nb_parallel.record_gemini_usage(resp, seconds)
    return LAST_USAGE

def gemini_json(system_prompt, user_message, model=None, temperature=0):
    global last_latency_s
    cfg = dict(system_instruction=system_prompt, temperature=temperature, response_mime_type="application/json")
    if MAX_OUTPUT_TOKENS is not None:
        cfg["max_output_tokens"] = MAX_OUTPUT_TOKENS
    if THINKING_BUDGET is not None and hasattr(genai_types, "ThinkingConfig"):
        cfg["thinking_config"] = genai_types.ThinkingConfig(thinking_budget=THINKING_BUDGET)
    t0 = time.perf_counter()
    resp = client.models.generate_content(model=model or GEMINI_MODEL, contents=user_message,
                                          config=genai_types.GenerateContentConfig(**cfg))
    last_latency_s = time.perf_counter() - t0
    _record_usage(resp, last_latency_s)                  # tokens + latency of THIS call
    return _loads_lenient(resp.text, resp)


# --- select the backend ---------------------------------------------------------------------
# Must stay LAST in this cell: the Gemini `gemini_json` above is redefined every time the cell
# runs, so whichever binding happens last wins. In a separate cell this would be silently undone.
# activate() sets gemini_json, GEMINI_MODEL, RESULTS_DIR, MAX_OUTPUT_TOKENS and THINKING_BUDGET.
# With BACKEND="gemini" it leaves the function above alone and only repoints RESULTS_DIR.
import sys, pathlib, importlib
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "llm_backend.py").exists():
        sys.path.insert(0, str(_p)); break
import nb_parallel, llm_backend
importlib.reload(nb_parallel); importlib.reload(llm_backend)   # plain `import` is a no-op once cached
nb_parallel.install()           # thread-aware stdout, so a parallel batch prints one block per article

llm_backend.activate(BACKEND, ROOT, globals(), results=RESULTS_MODEL)


Note: you may need to restart the kernel to use updated packages.
backend: vLLM http://localhost:8000/v1 model=qwen3-32b thinking=True structured=False max_tokens=None  ->  results/qwen3-32b-think/


{'backend': 'qwen3-32b-think',
 'kind': 'vllm',
 'model': 'qwen3-32b',
 'thinking': True,
 'structured': False,
 'results_dir': PosixPath('/Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/qwen3-32b-think')}

## 1. Single-call extraction — entities + n-ary nodes + relations, in one pass

One Gemini call takes the raw TEXT and returns everything: entities (with open free-form types and
coreference merged), n-ary nodes (Event / Observation / Claim), and open `relations`. The rules are the
same event-centric rules as nb6b, plus entity extraction and coreference in the same pass — which is
exactly the burden this baseline puts on a single call.

In [22]:
# E2E_SYSTEM_PROMPT = """You are an expert knowledge-graph extraction system. Given a news TEXT and its REFERENCE_DATE (ISO-8601), build a reified, event-centric knowledge graph in ONE pass, with THREE parts.

# (1) ENTITIES — the real-world things the TEXT mentions (people, organizations, places, quantities, dates, durations, etc.). Each entity: {"id" ("E1","E2",...), "text" (canonical surface form), "type" (a concise free-form PascalCase class label, as specific as the TEXT warrants, but a REUSABLE class with no specific named entity in the label), "aliases" (other surface forms for the SAME entity), "normalized_value", "normalized_end" (ISO-8601 for dates/times per the TEMPORAL rule; null otherwise), "confidence" (0-1, your certainty in the type)}. MERGE coreferent mentions into ONE entity; never duplicate an entity. Keep type labels consistent: the same kind of thing gets the exact same label.

# (2) N-ARY NODES — new reified nodes for the events, observations, and claims the TEXT is about. Kinds:
# - "Event": something that happens or is scheduled/forecast to happen.
# - "Observation": a stated measurement, statistic, quantity, or quantified change.
# - "Claim": a statement explicitly attributed to a source.
# Each n-ary node: {"id" ("EV1"/"OBS1"/"CL1",...), "node_kind", "type" (concise free-form PascalCase; reusable, no named entity in the label), "label" (a SHORT natural title for the event/observation/claim - a few words, no packed-in facts), "is_main" (bool), "evidence" (verbatim span(s)), "confidence" (0-1)}.

# (3) RELATIONS — open, free-form directed relations {"subject_id","relation_phrase","object_id","evidence","confidence"} where subject_id and object_id are ids of ENTITIES or N-ARY NODES. Choose a concise, GENERAL, reusable relation_phrase: prefer a short present-tense verb phrase (e.g. "located in", "attributed to", "caused by") that names the relation itself (canonicalized downstream) WITHOUT baking the object/subject into it; keep phrasing consistent (the same relation gets the same phrase) and orient each triple so subject --relation_phrase--> object reads correctly. Define each unique phrase in relation_definitions.

# RULES:
# 1. Identify the article's MAIN event(s) and set "is_main": true on them.
# 2. Tie every participant of a stated fact to its n-ary node with a relation, and capture the stable entity-to-entity relations. Aim for ONE connected graph — do not leave a relevant node dangling.
# 3. Keep every fact's context connected to it: attach all the details that qualify a fact — value, time, place, cause, qualifier, source — to THAT fact's n-ary node, so it forms one connected cluster reachable in a few hops.
# 4. Keep each n-ary node's label a SHORT, natural title (a few words, ~5 max) that reads as the event/observation/claim; do NOT pack facts into it - values, times, quantities, sources and participants are RELATIONS, not label text. Any entity you name in the label MUST also be attached as a relation (the label never replaces an edge).
# 5. Give every Observation and Claim at least one relation to the Event it describes, and relate each Event to other Events it is linked to in the TEXT if applicable (e.g. "part of" / "caused by" / "led to" / "followed").
# 6. When a breakdown gives several values that each belong to a distinct item (category, group, period, round), relate EACH value to its item AND link each item to the shared parent fact/event, so the whole breakdown stays in ONE connected cluster.
# 7. subject_id and object_id MUST be ids you defined in ENTITIES or N-ARY NODES; never relate a node to itself.
# 8. Extract every relevant entity; omit only boilerplate (navigation, "related"/"read more" links).
# 9. Extract only what the TEXT explicitly states or unambiguously entails; do NOT use outside knowledge. "evidence" is a list of VERBATIM, UNBROKEN spans copied character-for-character from the TEXT — never abbreviate, paraphrase, normalize punctuation, or join distant fragments with "…". Together the spans MUST contain the subject, the object, and the wording that expresses the relation (for an n-ary node, the wording that states the event/observation/claim). When a single sentence contains all of them, give that one sentence as the only span; when they are stated across different sentences, give EACH relevant sentence as its own separate span in the list rather than splicing them into one elided quote. Prefer the smallest set of COMPLETE sentences that fully supports the assertion. "confidence" (0-1) is lower when a fact, relation, or direction is implicit or ambiguous.
# 10. Normalize temporal entities: for any entity that denotes a date/time (absolute like "24 June", or relative/deictic like "Wednesday" / "the last week of the month" / "this weekend"), resolve it to an ISO-8601 "normalized_value" using REFERENCE_DATE and the narrative direction, at the granularity the text justifies; for a span also set "normalized_end" (inclusive end); for non-temporal or unresolvable entities set both to null. Never put the normalized value in "text".
# 11. Define every unique label: type_definitions: for each distinct entity "type", a general one-sentence "definition" + 2-5 "synonyms" (independent of the specific entities). relation_definitions: for each "relation_phrase", a NEUTRAL one-sentence "definition" (do not name subject/object or state a direction, so it reads the same for the relation and its inverse) + 2-5 "synonyms".

# OUTPUT strict JSON:
# {
#   "entities": [ {"id","text","type","aliases":[...],"normalized_value","normalized_end","confidence"} ],
#   "nary_nodes": [ {"id","node_kind","type","label","is_main","evidence":[...],"confidence"} ],
#   "relations": [ {"subject_id","relation_phrase","object_id","evidence":[...],"confidence"} ],
#   "type_definitions": [ {"type","definition","synonyms":[...]} ],
#   "relation_definitions": [ {"relation_phrase","definition","synonyms":[...]} ]
# }
# """

def _demojibake(s):
    if isinstance(s, str) and any(m in s for m in ("Ã", "Â", "â€")):
        try: return s.encode("latin-1").decode("utf-8")
        except (UnicodeDecodeError, UnicodeEncodeError): return s
    return s

def _reference_date(text):
    m = re.search(r"^Date:\s*(\d{2})/(\d{2})/(\d{4})", text, re.MULTILINE)
    if not m: return None
    d, mth, y = m.groups(); return f"{y}-{mth}-{d}"

def _fmt_usage(u):
    u = u or {}
    return (f"prompt {u.get('prompt_tokens')}, output {u.get('output_tokens')}, "
            f"thoughts {u.get('thoughts_tokens')}, total {u.get('total_tokens')}")

def extract_e2e(aid, write=True):
    """ONE Gemini call: raw TEXT -> entities + n-ary nodes + relations -> results/<id>/nary_graph_e2e.json."""
    t_article = time.perf_counter()
    text = _demojibake((DATASET_DIR / f"{aid}.txt").read_text())
    ref_date = _reference_date(text)
    out = gemini_json(E2E_SYSTEM_PROMPT, f"REFERENCE_DATE: {ref_date}\n\nTEXT:\n{text}")
    # per-THREAD usage, not the LAST_USAGE global: when the batch runs articles in parallel that
    # global holds whichever call finished last, which need not be this article's.
    ext_usage = nb_parallel.last_usage()                # capture NOW; the review call replaces it

    entities   = out.get("entities", [])
    nary_nodes = out.get("nary_nodes", [])
    relations  = out.get("relations", [])
    for e in entities:                                  # normalise: ensure aliases list exists
        e.setdefault("aliases", [])
    node_text = {e["id"]: (e.get("text"), e.get("type")) for e in entities}
    node_text.update({n["id"]: (n.get("label"), n.get("type")) for n in nary_nodes})
    for r in relations:                                 # resolve endpoints (so nb7b/nb8 read subject/object surface+type)
        r["subject_text"], r["subject_type"] = node_text.get(r.get("subject_id"), (None, None))
        r["object_text"], r["object_type"]   = node_text.get(r.get("object_id"), (None, None))

    graph = {"article_id": aid, "reference_date": ref_date, "method": "single_call_e2e",
             "entities": entities, "nary_nodes": nary_nodes,
             "relations": relations, "type_definitions": out.get("type_definitions", []),
             "relation_definitions": out.get("relation_definitions", [])}

    review_usage = None
    if REVIEW:                                          # optional 2nd LLM pass: check/repair vs TEXT
        graph = review_graph(aid, text, graph, ref_date)
        review_usage = nb_parallel.last_usage()          # the review call's usage

    # deterministic normalization: merge duplicate entities (esp. review-minted twins) + drop dup edges
    graph, dd = _dedup_graph(graph)
    entities, nary_nodes, relations = graph["entities"], graph["nary_nodes"], graph["relations"]
    if dd["entities_merged"] or dd["relations_dropped"] or dd["selfloops_dropped"]:
        print(f"    dedup {aid}: merged {dd['entities_merged']} dup entities, "
              f"dropped {dd['relations_dropped']} dup + {dd['selfloops_dropped']} self-loop relations")

    # record per-call token usage on the graph (persisted into the written JSON) + print it
    def _tot(u, k): return (u or {}).get(k) or 0
    combined = {k: _tot(ext_usage, k) + _tot(review_usage, k)
                for k in ("prompt_tokens", "output_tokens", "thoughts_tokens", "total_tokens")}
    graph["token_usage"] = {"extraction": ext_usage, "review": review_usage, "combined": combined}

    if write:
        p = RESULTS_DIR / aid; p.mkdir(parents=True, exist_ok=True)
        (p / OUTPUT_NAME).write_text(json.dumps(graph, indent=2, ensure_ascii=False))
        print(f"  wrote {p / OUTPUT_NAME}  ({len(entities)} entities, {len(nary_nodes)} n-ary nodes, "
              f"{len(relations)} relations)  [{time.perf_counter() - t_article:.1f}s]")
    print(f"  tokens — extraction: {_fmt_usage(ext_usage)}"
          + (f"  |  review: {_fmt_usage(review_usage)}" if review_usage else "")
          + f"  |  article total: {combined['total_tokens']}")
    return graph

In [23]:
# # --- EXPERIMENTAL alternative prompt: HOLISTIC (not decomposed into 3 separate tasks) ---
# # Same rules + same OUTPUT schema as the 3-part prompt above (so extract_e2e / 6d / 6e / nb8 all still
# # work), but framed as building ONE reified, event-centric KG directly instead of "extract entities,
# # THEN n-ary nodes, THEN relations". Run THIS cell to use it; re-run the cell above to go back to the
# # 3-part prompt. (extract_e2e reads the global E2E_SYSTEM_PROMPT at call time.)
# # NOTE: outputs still overwrite the same OUTPUT_NAME (nary_graph_e2e.json). To keep both variants,
# # set OUTPUT_NAME (in the config cell) to e.g. "e2e/nary_graph_holistic.json" before running the batch.
# E2E_SYSTEM_PROMPT_HOLISTIC = """You are an expert knowledge-graph extraction system. Given a news TEXT and its REFERENCE_DATE (ISO-8601), represent everything the TEXT states as ONE reified, event-centric knowledge graph, built directly in a single pass.

# In this graph the news FACTS are first-class reified nodes and the real-world entities are wired to them:
# - Reified fact nodes (n-ary nodes) stand for the events, observations, and claims the TEXT is about \u2014 an "Event" (something that happens or is scheduled/forecast to happen), an "Observation" (a stated measurement, statistic, quantity, or quantified change), or a "Claim" (a statement explicitly attributed to a source). Each has a SHORT natural title (a few words, ~5 max) that reads as the event/observation/claim, with no packed-in facts - values, times, sources and participants are relations, not label text; any entity named in a label must also be attached as a relation.
# - Entities are the people, organizations, places, quantities, dates, durations, etc. Coreferent mentions are MERGED into ONE entity, never duplicated.
# - Relations are open, free-form directed edges (invent the concise, GENERAL, reusable relation_phrase that names the relation itself (without baking the object/subject into it) \u2014 canonicalized downstream) between any two ids, entity or reified node, that connect every fact to its participants and context and connect facts to one another, so the whole graph is ONE connected component.

# Build the graph so it satisfies:
# 1. The article's MAIN fact(s) carry "is_main": true.
# 2. Every participant and every qualifying detail of a stated fact \u2014 value, time, place, cause, qualifier, source \u2014 attaches to THAT fact's reified node as a relation, forming one connected cluster reachable in a few hops; also capture the stable entity-to-entity relations, and leave no relevant node dangling.
# 3. Keep each reified node's label a SHORT handle; express values, places, times, causes and participants as RELATIONS to entities, not inside the label.
# 4. Every Observation and Claim has at least one relation to the Event it describes; relate each Event to other Events it is linked to in the TEXT (e.g. "part of" / "caused by" / "led to" / "followed").
# 5. When a breakdown gives several values that each belong to a distinct item (category, group, period), relate EACH value directly to the item it belongs to (value -> item).
# 6. Every subject_id/object_id MUST be an id you defined; never relate a node to itself.
# 7. Include every relevant entity; omit only boilerplate (navigation, "related"/"read more" links).
# 8. Extract only what the TEXT explicitly states or unambiguously entails; do NOT use outside knowledge. "evidence" is a list of VERBATIM, UNBROKEN spans copied character-for-character from the TEXT — never abbreviate, paraphrase, normalize punctuation, or join distant fragments with "…". Together the spans MUST contain the subject, the object, and the wording that expresses the relation (for an n-ary node, the wording that states the event/observation/claim). When a single sentence contains all of them, give that one sentence as the only span; when they are stated across different sentences, give EACH relevant sentence as its own separate span in the list rather than splicing them into one elided quote. Prefer the smallest set of COMPLETE sentences that fully supports the assertion. "confidence" (0-1) is lower when a fact, relation, or direction is implicit or ambiguous.
# - Temporal normalization: for any entity that denotes a date/time (absolute like "24 June", or relative/deictic like "Wednesday" / "the last week of the month" / "this weekend"), resolve it to an ISO-8601 "normalized_value" using REFERENCE_DATE and the narrative direction, at the granularity the text justifies; for a span also set "normalized_end" (inclusive end); for non-temporal or unresolvable entities set both to null. Never put the normalized value in "text".
# - Definitions: after extraction, define every unique label. type_definitions: for each distinct entity "type", a general one-sentence "definition" + 2-5 "synonyms" (independent of the specific entities). relation_definitions: for each "relation_phrase", a NEUTRAL one-sentence "definition" (do not name subject/object or state a direction, so it reads the same for the relation and its inverse) + 2-5 "synonyms".

# Serialize the graph as strict JSON with these collections:
# {
#   "entities": [ {"id" ("E1",...), "text" (canonical surface form), "type" (concise free-form PascalCase class; reusable, no named entity in the label), "aliases":[...], "normalized_value", "normalized_end", "confidence" (0-1)} ],
#   "nary_nodes": [ {"id" ("EV1"/"OBS1"/"CL1",...), "node_kind", "type" (concise free-form PascalCase; reusable, no named entity in the label), "label" (SHORT natural title; no packed-in facts), "is_main" (bool), "evidence":[verbatim span(s)], "confidence" (0-1)} ],
#   "relations": [ {"subject_id","relation_phrase","object_id","evidence":[...],"confidence"} ],
#   "type_definitions": [ {"type","definition","synonyms":[...]} ],
#   "relation_definitions": [ {"relation_phrase","definition","synonyms":[...]} ]
# }
# """

# # Activate the holistic prompt (comment out the next line to keep the 3-part prompt above active):
# E2E_SYSTEM_PROMPT = E2E_SYSTEM_PROMPT_HOLISTIC
# print("E2E prompt: HOLISTIC (single reified-KG instruction) is now ACTIVE  \u2014 re-run the previous cell to revert to the 3-part prompt")


In [24]:
# --- EXPERIMENTAL prompt v4: 3-PART + DEDUP + aboutness / anti-over-reification ---
# Builds on the 3-part dedup prompt. Reorganized for readability (bulleted field schemas + grouped
# rules). Rules ADDED (these change behavior vs the earlier dedup prompt):
#   * Aboutness: every n-ary node — and each main Event — must link to the core subject/topic entity.
#   * Reification discipline: prefer a direct entity-to-entity relation over a Claim/Observation when it
#     is clearer/more logical (don't over-reify stable, definitional facts).
#   * Entity materialization: if a relation needs a subject/object not yet in ENTITIES (incl. anything
#     named only inside an n-ary label), create that entity and use its id.
#   * Final review/repair is now a SEPARATE 2nd LLM call (review_graph below) — NOT part of this prompt.
# Run THIS cell to use it; re-run the 3-part cell (top) or the holistic cell to switch.
# NOTE: still writes OUTPUT_NAME. Set OUTPUT_NAME (config cell) to e.g. "e2e/nary_graph_dedup.json"
# before the batch to keep this variant separate.
E2E_SYSTEM_PROMPT_DEDUP = """You are an expert knowledge-graph extraction system. Given a news TEXT and its REFERENCE_DATE (ISO-8601), build a reified, event-centric knowledge graph in one pass, with three parts.

PART 1 - ENTITIES
The real-world things the TEXT mentions. Each entity:
- "id": "E1","E2",...
- "text": canonical surface form
- "type": a concise free-form PascalCase class label, as specific as the TEXT warrants, but a reusable class with no specific named entity in the label
- "aliases": other surface forms for the same entity
- "normalized_value" / "normalized_end": ISO-8601 for dates/times per the Temporal normalization rule; null otherwise
- "confidence": 0-1, your certainty in the type

Merge coreferent mentions into one entity; never duplicate an entity. Keep type labels consistent: the same kind of thing gets the exact same label.

PART 2 - N-ARY NODES
New reified nodes for the events and claims the TEXT is about. Kinds:
- "Event": something that happens or is scheduled/forecast to happen.
- "Claim": a statement explicitly attributed to a source.

Each n-ary node:
- "id": "EV1"/"OBS1"/"CL1",...
- "node_kind": Event or Claim
- "type": concise free-form PascalCase; reusable, no named entity in the label
- "label": a short, natural title
- "is_main": bool
- "evidence": verbatim span(s)
- "confidence": 0-1

PART 3 - RELATIONS
Open, free-form directed relations. Each relation:
- "subject_id" / "object_id": ids of ENTITIES or N-ARY NODES
- "relation_phrase": concise, general, and reusable; without baking the object/subject into it; orient each triple so subject --relation--> object reads correctly. Can be a verb or a noun.
- "evidence": verbatim span(s)
- "confidence": 0-1

RULES:

Main event & event structure:
- Identify the article's main event(s) and set "is_main": true on them.
- Give every Claim at least one relation to the Event it is related to, and relate each Event to other Events it is linked to in the TEXT if applicable (e.g. "part of" / "caused by" / "led to" / "followed").

Aboutness:
- Every n-ary node must link to the core subject/topic entity it concerns via an "about" relation. In particular, connect each main Event to the primary entity the article centres on (not an n-ary node), so that central entity is not left stranded.

Reification discipline (do not over-reify):
- When a fact is a stable, direct relationship between two entities and a direct entity-to-entity relation expresses it clearly and logically, use the direct relation instead of creating an n-ary node. Reserve Event/Claim nodes for genuine events and attributed statements.

Deduplication:
- Deduplicate n-ary nodes: never emit two nodes for the same event, or claim. When a fact appears more than once (restated, elaborated, described from another angle, or attributed again), keep a single surviving node and fold every connected entity, value, time, place, cause, qualifier, source and relation from all the duplicate mentions onto that one node, unioning their evidence spans — so no participant or detail is lost. This also applies across node kinds: if an event/claim you reified as an n-ary node would also appear as a standalone entity, do NOT create that separate ENTITY — use the n-ary node's id as the relation argument (if one is created anyway, link it "sameAs" the node).

Connectivity & context:
- Tie every participant of a stated fact to its n-ary node with a relation, and capture all the the entity-to-entity relations mentioned in text. Aim for one connected graph — do not leave a relevant node dangling.
- Every subject_id and object_id MUST resolve to an id that exists in ENTITIES or N-ARY NODES.

Labels:
- Keep each n-ary node's label a short, natural title (a few words, ~5 max) that reads as the event/claim; do NOT pack facts into it - values, times, quantities, sources and participants are RELATIONS, not label text. Any entity you name in the label MUST also be attached as a relation (the label never replaces an edge).

Value breakdowns:
- When a breakdown gives several values that each belong to a distinct item (a category, group, period, rank, etc), attach each value directly to its own item (item --relation--> value), then link each item to the shared parent fact/event. Do NOT attach both the values and the items directly to the parent as separate edges — that loses which value pairs with which item (i.e. item --relation--> its own value, NOT parent --relation--> every value alongside parent --relation--> every item).

Coverage & faithfulness:
- Extract every relevant entity; omit only boilerplate (navigation, "related"/"read more" links).
- Extract only what the TEXT explicitly states or unambiguously entails; do NOT use outside knowledge. 
- "evidence": Verbatim, complete sentences copied character-for-character, as a list with one span per sentence. Together they must explicitly mention the subject, object and the relation (for an n-ary node, the event/claim). The evidence will be used for grounding the fact later.
- "confidence" (0-1) is lower when a fact, relation, or direction is implicit or ambiguous.

Temporal normalization:
- Normalize temporal entities: for any entity that denotes a date/time (absolute like "24 June", or relative/deictic like "Wednesday" / "the last week of the month" / "this weekend"), resolve it to an ISO-8601 "normalized_value" using REFERENCE_DATE and the narrative direction, at the granularity the text justifies; for a span also set "normalized_end" (inclusive end); for non-temporal or unresolvable entities set both to null. Never put the normalized value in "text".

Definitions:
- Define every unique label: type_definitions: for each distinct entity "type", a general one-sentence "definition" + 2-5 "synonyms" (independent of the specific entities). relation_definitions: for each "relation_phrase", a neutral one-sentence "definition" (do not name subject/object or state a direction, so it reads the same for the relation and its inverse) + 2-5 "synonyms".

OUTPUT strict JSON:
{
  "entities": [ {"id","text","type","aliases":[...],"normalized_value","normalized_end","confidence"} ],
  "nary_nodes": [ {"id","node_kind","type","label","is_main","evidence":[...],"confidence"} ],
  "relations": [ {"subject_id","relation_phrase","object_id","evidence":[...],"confidence"} ],
  "type_definitions": [ {"type","definition","synonyms":[...]} ],
  "relation_definitions": [ {"relation_phrase","definition","synonyms":[...]} ]
}
"""

#- Keep every fact's context connected to it: attach all the details that qualify a fact — value, time, place, cause, qualifier, source — to that fact's n-ary node, so it forms one connected cluster reachable in a few hops.
#- "evidence" is a list of VERBATIM, UNBROKEN spans copied character-for-character from the TEXT — never abbreviate, paraphrase, normalize punctuation, or join distant fragments with "…". Together the spans MUST contain the subject, the object, and the wording that expresses the relation (for an n-ary node, the wording that states the event/claim). When a single sentence contains all of them, give that one sentence as the only span; when they are stated across different sentences, give EACH relevant sentence as its own separate span in the list rather than splicing them into one elided quote. Prefer the smallest set of COMPLETE sentences that fully supports the assertion. 


# Activate the dedup prompt (comment out to keep whichever prompt is currently active):
E2E_SYSTEM_PROMPT = E2E_SYSTEM_PROMPT_DEDUP
print("E2E prompt: 3-PART + DEDUP + aboutness/anti-over-reify is now ACTIVE  — re-run the 3-part or holistic cell to switch")

E2E prompt: 3-PART + DEDUP + aboutness/anti-over-reify is now ACTIVE  — re-run the 3-part or holistic cell to switch


In [25]:
# ── OPTIONAL review/repair pass (SEPARATE 2nd LLM call), replaces the old in-prompt PART 4 ───────────
# Takes the TEXT + the draft graph and returns a small PATCH (not a full rewrite): it materialises any
# referenced-but-missing endpoint entity, adds relations the TEXT states but the draft missed, collapses
# an over-reified n-ary node into a direct entity-to-entity edge, drops redundant/incorrect edges, and
# repairs evidence that fails to name its subject/object (a coreferent stand-in like "some areas").
# Returning a PATCH (rather than the whole graph) means the pass can only touch what it names — it cannot
# silently drop the rest. A deterministic guard then removes any relation whose endpoint is still
# undefined, so the written graph is always self-consistent. Set REVIEW=False to skip the pass.
REVIEW = True

REVIEW_PROMPT = """You are an expert knowledge-graph reviewer and repairer. You are given a news TEXT and a DRAFT graph (entities, nary_nodes, relations) already extracted from it. Do NOT rebuild the graph. Inspect it against the TEXT and return a small PATCH that fixes ONLY these problems:

1. MISSING FACTS — add any relation the TEXT states or unambiguously entails that the draft omitted (entity-to-entity or entity-to-n-ary-node), plus any entity a relation needs but that is missing with its surface text and a concise reusable PascalCase type. Every subject_id/object_id MUST resolve to a defined id (reuse the referenced id when materialising a missing one, else a fresh id). Never invent a relation the TEXT does not support.
2. OVER-REIFICATION — if an n-ary node encodes what is really a stable, direct fact between two entities, collapse it: list the node id in remove_nary_nodes and re-express the fact as a direct entity-to-entity relation in add_relations (carry its evidence). Only when no information is lost.
3. REDUNDANT / WRONG edges — remove exact-duplicate edges, and fix a wrong direction (remove the wrong one and add the corrected one).
4. EVIDENCE GAPS — every relation's and n-ary node's evidence spans must EXPLICITLY name its subject and object (for a node, every entity named in its label); a pronoun or definite description ("this", "the country", "some areas") does NOT count. Where the naming sentence is missing, return corrected evidence — the original spans plus the nearest VERBATIM complete sentence(s) that name the entity — via fix_relation_evidence / fix_nary_node_evidence.

Keep every existing correct fact; do not remove anything unless it is redundant or you are re-expressing it. Return ONLY valid JSON — a PATCH, using empty lists where there is nothing to change:
{
  "add_entities":      [ {"id","text","type","aliases":[],"normalized_value":null,"normalized_end":null,"confidence":1.0} ],
  "add_nary_nodes":    [ {"id","node_kind","type","label","is_main":false,"evidence":[],"confidence":1.0} ],
  "remove_nary_nodes": [ "<id>" ],
  "add_relations":     [ {"subject_id","relation_phrase","object_id","evidence":[],"confidence":1.0} ],
  "remove_relations":  [ {"subject_id","relation_phrase","object_id"} ],
  "fix_relation_evidence":  [ {"subject_id","relation_phrase","object_id","evidence":[]} ],
  "fix_nary_node_evidence": [ {"id","evidence":[]} ],
  "add_relation_definitions": [ {"relation_phrase","definition","synonyms":[]} ],
  "add_type_definitions":     [ {"type","definition","synonyms":[]} ],
  "notes": "one short line on what you changed"
}"""


def _resolve_endpoint_texts(graph):
    """Recompute subject/object text+type on every relation from the current entities + n-ary nodes."""
    ref = {e["id"]: (e.get("text"), e.get("type")) for e in graph["entities"]}
    ref.update({n["id"]: (n.get("label"), n.get("type")) for n in graph["nary_nodes"]})
    for r in graph["relations"]:
        r["subject_text"], r["subject_type"] = ref.get(r.get("subject_id"), (None, None))
        r["object_text"],  r["object_type"]  = ref.get(r.get("object_id"),  (None, None))
    return graph

def _guard_endpoints(graph):
    """Drop any relation whose subject or object id is still undefined (last-resort self-consistency)."""
    ids = {e["id"] for e in graph["entities"]} | {n["id"] for n in graph["nary_nodes"]}
    kept, dropped = [], []
    for r in graph["relations"]:
        (kept if r.get("subject_id") in ids and r.get("object_id") in ids else dropped).append(r)
    graph["relations"] = kept
    return graph, dropped

def _union_evidence(old, new):
    """Append new spans not already present — augment evidence, never shrink it, so an EVIDENCE-GAP fix
    can only ADD the missing naming sentence even if the model forgets to echo the original spans."""
    merged = list(old or [])
    for s in (new or []):
        if s not in merged:
            merged.append(s)
    return merged

import unicodedata
def _norm(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", s or "")).strip().casefold()

def _dedup_graph(graph):
    """Enforce the extraction's 'never duplicate an entity' rule deterministically. The review pass
    sometimes mints a FRESH id for an entity that already exists (E88 'Europe' alongside E3 'Europe'),
    which fragments a node's edges and strands the twin. Merge entities sharing
    (normalized text, normalized type, normalized_value) onto the first (extraction) id; remap every
    relation endpoint; then drop exact-duplicate edges (union evidence) and any self-loop a merge made.
    Returns (graph, stats)."""
    remap, canon, keep = {}, {}, []
    for e in graph.get("entities", []):
        key = (_norm(e.get("text")), _norm(e.get("type")), e.get("normalized_value"))
        if not key[0]:                                   # no surface text -> cannot dedup, keep as-is
            keep.append(e); continue
        c = canon.get(key)
        if c is None:
            canon[key] = e; keep.append(e)
        else:                                            # twin -> fold into the canonical entity
            remap[e["id"]] = c["id"]
            al = {a for a in (c.get("aliases") or [])} | {a for a in (e.get("aliases") or [])}
            if e.get("text"): al.add(e["text"])
            c["aliases"] = sorted(a for a in al if a and _norm(a) != _norm(c.get("text")))
            if not c.get("normalized_value") and e.get("normalized_value"):
                c["normalized_value"] = e.get("normalized_value"); c["normalized_end"] = e.get("normalized_end")
    graph["entities"] = keep

    seen, newrels, selfloops = {}, [], 0
    for r in graph.get("relations", []):
        r["subject_id"] = remap.get(r.get("subject_id"), r.get("subject_id"))
        r["object_id"]  = remap.get(r.get("object_id"),  r.get("object_id"))
        if r["subject_id"] == r["object_id"]:            # a merge can collapse an edge onto itself
            selfloops += 1; continue
        k = (r.get("subject_id"), r.get("relation_phrase") or r.get("property"), r.get("object_id"))
        if k in seen:
            seen[k]["evidence"] = _union_evidence(seen[k].get("evidence"), r.get("evidence"))
        else:
            seen[k] = r; newrels.append(r)
    dropped = len(graph.get("relations", [])) - len(newrels) - selfloops
    graph["relations"] = newrels
    _resolve_endpoint_texts(graph)                       # endpoint texts may have changed via remap
    return graph, {"entities_merged": len(remap), "relations_dropped": dropped, "selfloops_dropped": selfloops}

def _apply_patch(graph, patch):
    ents  = {e["id"]: e for e in graph["entities"]}
    nodes = {n["id"]: n for n in graph["nary_nodes"]}
    for e in (patch.get("add_entities") or []):
        e.setdefault("aliases", []); ents.setdefault(e.get("id"), e)     # never overwrite an existing card
    for n in (patch.get("add_nary_nodes") or []):
        nodes.setdefault(n.get("id"), n)
    rm = set(patch.get("remove_nary_nodes") or [])
    for nid in rm:
        nodes.pop(nid, None)
    rmrel = {(x.get("subject_id"), x.get("relation_phrase"), x.get("object_id"))
             for x in (patch.get("remove_relations") or [])}
    def _key(r): return (r.get("subject_id"), r.get("relation_phrase") or r.get("property"), r.get("object_id"))
    rels = [r for r in graph["relations"]                                 # drop edges on removed nodes + explicit removals
            if r.get("subject_id") not in rm and r.get("object_id") not in rm and _key(r) not in rmrel]
    rels += (patch.get("add_relations") or [])
    # EVIDENCE GAPS (rule 4): augment the evidence of an existing relation / node so its subject+object
    # are explicitly named. Match a relation by (subject, phrase, object), a node by id; union so the
    # fix can only ADD the naming sentence, never drop the original spans.
    rel_by_key = {}
    for r in rels:
        rel_by_key.setdefault(_key(r), r)
    for fx in (patch.get("fix_relation_evidence") or []):
        r = rel_by_key.get((fx.get("subject_id"), fx.get("relation_phrase"), fx.get("object_id")))
        if r is not None:
            r["evidence"] = _union_evidence(r.get("evidence"), fx.get("evidence"))
    for fx in (patch.get("fix_nary_node_evidence") or []):
        n = nodes.get(fx.get("id"))
        if n is not None:
            n["evidence"] = _union_evidence(n.get("evidence"), fx.get("evidence"))
    rdefs = graph.get("relation_definitions", []); seen_r = {d.get("relation_phrase") for d in rdefs}
    rdefs += [d for d in (patch.get("add_relation_definitions") or []) if d.get("relation_phrase") not in seen_r]
    tdefs = graph.get("type_definitions", []);     seen_t = {d.get("type") for d in tdefs}
    tdefs += [d for d in (patch.get("add_type_definitions") or []) if d.get("type") not in seen_t]
    graph["entities"] = list(ents.values()); graph["nary_nodes"] = list(nodes.values())
    graph["relations"] = rels; graph["relation_definitions"] = rdefs; graph["type_definitions"] = tdefs
    return graph

def review_graph(aid, text, graph, ref_date=None):
    """SECOND Gemini call: TEXT + draft graph -> PATCH -> apply -> guard -> resolve endpoint texts."""
    before  = (len(graph["entities"]), len(graph["nary_nodes"]), len(graph["relations"]))
    payload = {k: graph[k] for k in ("entities", "nary_nodes", "relations")}
    patch = gemini_json(REVIEW_PROMPT, f"TEXT:\n{text}\n\nDRAFT GRAPH (JSON):\n"
                        + json.dumps(payload, ensure_ascii=False))
    if not isinstance(patch, dict):
        print(f"    [warn] {aid}: review returned no patch object; keeping draft")
        return _resolve_endpoint_texts(graph)
    graph = _apply_patch(graph, patch)
    graph, dropped = _guard_endpoints(graph)
    graph = _resolve_endpoint_texts(graph)
    graph["review_notes"] = patch.get("notes")
    after = (len(graph["entities"]), len(graph["nary_nodes"]), len(graph["relations"]))
    n_fix = len(patch.get("fix_relation_evidence") or []) + len(patch.get("fix_nary_node_evidence") or [])
    print(f"    review {aid}: entities {before[0]}->{after[0]}  nodes {before[1]}->{after[1]}  rels {before[2]}->{after[2]}  "
          f"(+{len(patch.get('add_entities') or [])} ent, +{len(patch.get('add_relations') or [])} rel, "
          f"collapsed {len(patch.get('remove_nary_nodes') or [])} node, fixed {n_fix} ev, guard-dropped {len(dropped)})")
    return graph


## 2. Inspect the demo article

In [29]:
# DEMO — one article, prints the graph. COMMENTED OUT so a batch run does not extract
# ARTICLE_ID twice (the batch cell below covers every article in SELECTED_ARTICLES).
# Uncomment to inspect a single article interactively.
# g = extract_e2e(ARTICLE_ID)
# by_id = {e["id"]: e.get("text") for e in g["entities"]}
# by_id.update({n["id"]: n["label"] for n in g["nary_nodes"]})
# print("-" * 80)
# print(f'{len(g["entities"])} entities  |  {len(g["nary_nodes"])} n-ary nodes  |  {len(g["relations"])} relations')
# print("-" * 80)
# print("n-ary nodes:")
# for n in g["nary_nodes"]:
#     star = " *MAIN*" if n.get("is_main") else ""
#     print(f'  {n["id"]:<5} {n["node_kind"]:<12} <{n.get("type")}>{star}  "{n["label"]}"')
# print("-" * 80)
# print("relations (first 30):")
# for r in g["relations"][:30]:
#     s = by_id.get(r["subject_id"], r["subject_id"]); o = by_id.get(r["object_id"], r["object_id"])
#     print(f'  {s!r}  --{r["relation_phrase"]}-->  {o!r}')

APITimeoutError: Request timed out.

## 3. Batch — process the selected articles (`ARTICLE_IDS`)

In [27]:
# Single-call extraction for each article. Re-runs overwrite. Compare against the staged nary_graph.json.
# WORKERS articles at a time (set WORKERS=1 in the config cell for the plain sequential loop). Each
# article's prints are buffered and shown as one block when it finishes; a failure is reported and
# skipped rather than killing the batch.
nb_parallel.reset_usage()          # the usage report at the end covers only THIS run
_res  = nb_parallel.pmap(extract_e2e, ARTICLE_IDS, workers=WORKERS)
_runs = [(r.item, r.value) for r in _res if r.ok]

# --- token usage summary (extraction + review calls) ------------------------------------------------
def _u(g, which, k): return ((g.get("token_usage", {}) or {}).get(which) or {}).get(k) or 0
print("\n" + "=" * 60, "TOKEN USAGE")
print(f'{"article":8} {"extract_in":>11} {"extract_out":>12} {"review_in":>10} {"review_out":>11} '
      f'{"thoughts":>9} {"total":>8}')
for aid, g in _runs:
    print(f'{aid:8} {_u(g,"extraction","prompt_tokens"):>11} {_u(g,"extraction","output_tokens"):>12} '
          f'{_u(g,"review","prompt_tokens"):>10} {_u(g,"review","output_tokens"):>11} '
          f'{_u(g,"combined","thoughts_tokens"):>9} {_u(g,"combined","total_tokens"):>8}')
_grand = {k: sum(_u(g, "combined", k) for _, g in _runs)
          for k in ("prompt_tokens", "output_tokens", "thoughts_tokens", "total_tokens")}
print(f'{"TOTAL":8} {"":>11} {"":>12} {"":>10} {"":>11} {_grand["thoughts_tokens"]:>9} {_grand["total_tokens"]:>8}')
print(f'  grand total: prompt {_grand["prompt_tokens"]}, output {_grand["output_tokens"]}, '
      f'thoughts {_grand["thoughts_tokens"]}, total {_grand["total_tokens"]}')

# persist alongside the graphs (distinct name so it never clobbers 6e's token_usage_<variant>.json)
_tok_path = RESULTS_DIR / "_global" / "e2e" / "token_usage_extraction_dedup.json"
_tok_path.write_text(json.dumps(
    {"method": "single_call_e2e", "model": GEMINI_MODEL, "thinking_budget": THINKING_BUDGET,
     "per_article": [{"article": aid, **(g.get("token_usage", {}) or {})} for aid, g in _runs],
     "totals": _grand}, indent=2, ensure_ascii=False))
print(f"\nwrote {_tok_path}")
print(f"done -- {len(_runs)}/{len(ARTICLE_IDS)} article(s) -> results/{RESULTS_DIR.name}/<id>/{OUTPUT_NAME}")

# --- what this run cost: calls, prompt / output / THINKING tokens, and elapsed time ------------
# wall_seconds is the batch's real clock; with WORKERS>1 the summed LLM time is larger, and the
# ratio between them is the overlap the thread pool actually bought.
nb_parallel.usage_report("6c · e2e extraction", wall_seconds=nb_parallel.LAST_WALL_SECONDS)

4 item(s), running on 4 threads



[1/4] art4  (291.0s)
       review art4: entities 22->29  nodes 5->5  rels 21->27  (+7 ent, +8 rel, collapsed 0 node, fixed 1 ev, guard-dropped 0)
     wrote /Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/qwen3-32b-think/art4/e2e/nary_graph_dedup.json  (29 entities, 5 n-ary nodes, 27 relations)  [291.0s]
     tokens — extraction: prompt 2332, output 5009, thoughts 1146, total 7341  |  review: prompt 5179, output 1893, thoughts 830, total 7072  |  article total: 14413
[2/4] art2  (676.7s)
       review art2: entities 22->26  nodes 7->7  rels 26->31  (+4 ent, +7 rel, collapsed 0 node, fixed 6 ev, guard-dropped 1)
     wrote /Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/qwen3-32b-think/art2/e2e/nary_graph_dedup.json  (26 entities, 7 n-ary nodes, 31 relations)  [676.7s]
     tokens — extraction: prompt 3191, output 5653, thoughts 931, total 8844  |  review: prompt 6883, output 9550, thoughts 9075, total 

{'label': '6c · e2e extraction',
 'totals': {'calls': 6,
  'prompt_tokens': 29376,
  'output_tokens': 40050,
  'thoughts_tokens': 21262,
  'total_tokens': 69426,
  'seconds': 1755.3},
 'wall_seconds': 1801.3059291669633,
 'per_tag': {'art4': {'calls': 2,
   'seconds': 291.0,
   'prompt_tokens': 7511,
   'output_tokens': 6902,
   'thoughts_tokens': 1976,
   'total_tokens': 14413},
  'art2': {'calls': 2,
   'seconds': 676.7,
   'prompt_tokens': 10074,
   'output_tokens': 15203,
   'thoughts_tokens': 10006,
   'total_tokens': 25277},
  'art5': {'calls': 2,
   'seconds': 787.7,
   'prompt_tokens': 11791,
   'output_tokens': 17945,
   'thoughts_tokens': 9280,
   'total_tokens': 29736}},
 'calls': [{'prompt_tokens': 2332,
   'output_tokens': 5009,
   'thoughts_tokens': 1146,
   'total_tokens': 7341,
   'seconds': 199.16192258300725,
   'tag': 'art4'},
  {'prompt_tokens': 3191,
   'output_tokens': 5653,
   'thoughts_tokens': 931,
   'total_tokens': 8844,
   'seconds': 230.86295183293987,
   '